In [1]:
!nvidia-smi

Sun May 10 17:09:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile matrix_mul.cu

#include <stdio.h>
#include <cuda_runtime.h>

__global__ void matrixMul(int *A, int *B, int *C)
{
    int row = threadIdx.y;
    int col = threadIdx.x;

    int sum = 0;

    for(int k = 0; k < 2; k++)
    {
        sum += A[row * 2 + k] * B[k * 2 + col];
    }

    C[row * 2 + col] = sum;
}

int main()
{
    int h_A[4] = {1, 2,
                  3, 4};

    int h_B[4] = {5, 6,
                  7, 8};

    int h_C[4];

    int *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, 4 * sizeof(int));
    cudaMalloc((void**)&d_B, 4 * sizeof(int));
    cudaMalloc((void**)&d_C, 4 * sizeof(int));

    cudaMemcpy(d_A, h_A, 4 * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, 4 * sizeof(int), cudaMemcpyHostToDevice);

    dim3 threads(2, 2);

    matrixMul<<<1, threads>>>(d_A, d_B, d_C);

    cudaMemcpy(h_C, d_C, 4 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("Matrix Multiplication Result:\n");

    for(int i = 0; i < 4; i++)
    {
        printf("%d ", h_C[i]);

        if((i + 1) % 2 == 0)
            printf("\n");
    }

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing matrix_mul.cu


In [3]:
!nvcc -o matrix_mul matrix_mul.cu

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [4]:
!./matrix_mul

Matrix Multiplication Result:
19 22 
43 50 
